# 🔬 AUTSL 10 Sınıf: Pose-Only (Sadece İskelet) Teşhis Deneyi
Bu notebook, **Seed 7** ve merkez onarımlı önbellek (`cache/autsl_10_features_centerrepair_strict`) üzerinde **RGB el kırpımlarını kapatıp**, modeli **sadece 225 iskelet noktasıyla (MediaPipe koordinatları)** eğitir.

> **Bu Deneyin Temel Amacı Nedir?**
> * **İşaretçi 11'in (Signer 11)** kürk yelek ve koyu kıyafet içeren 4 hatalı videosunun **RGB fotoğraflarından mı** yoksa **iskelet noktalarından mı** kaynaklandığını bilimsel olarak teşhis etmektir.
> * Eğer model el görüntüleri olmadan (sadece iskeletle) İşaretçi 11'i doğru bilirse: Hatanın görüntüdeki kürk dokusundan kaynaklandığı kanıtlanır.
> * Eğitim süresi: CNN omurgası devre dışı olduğu için T4 GPU ile sadece **~4 dakika** sürer.

### 1. Google Drive Bağlantısı ve Gerekli Kütüphaneler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Gerekli kütüphaneleri yükle
!pip install -q pyyaml rich opencv-python matplotlib scikit-learn tqdm mediapipe

import os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("✅ Google Drive bağlandı, bağımlılıklar yüklendi.")

### 2. Proje Kodlarının ve Config Dosyalarının Yüklenmesi
*Drive'a attığınız `colab_kod_guncelleme.zip` dosyasını otomatik bulup Colab yerel SSD'sine açar.*

In [ ]:
import glob, os, shutil

local_project = "/content/BitirmeProjesi"
os.makedirs(local_project, exist_ok=True)

# 1. Drive'daki colab_kod_guncelleme.zip dosyasını nerede olursa olsun otomatik bul
found_zip = glob.glob("/content/drive/MyDrive/**/colab_kod_guncelleme.zip", recursive=True)

if found_zip:
    zip_path = found_zip[0]
    print(f"📦 Kod paketi bulundu: {zip_path}")
    !unzip -q -o "{zip_path}" -d {local_project}
    print("✅ Kodlar ve yeni configler başarıyla açıldı!")
else:
    print("⚠️ colab_kod_guncelleme.zip bulunamadı. Drive'daki klasörlerden kopyalanıyor...")
    drive_project = "/content/drive/MyDrive/BitirmeProjesi"
    for folder in ["src", "scripts", "configs"]:
        src_f = os.path.join(drive_project, folder)
        dst_f = os.path.join(local_project, folder)
        if os.path.exists(src_f):
            if os.path.exists(dst_f):
                shutil.rmtree(dst_f)
            shutil.copytree(src_f, dst_f)
            print(f"✅ {folder} kopyalandı.")

# Checkpoint ve logların kalıcı olması için outputs klasörünü Drive'a bağla
drive_outputs = "/content/drive/MyDrive/BitirmeProjesi/outputs"
os.makedirs(drive_outputs, exist_ok=True)
if not os.path.exists(f"{local_project}/outputs"):
    os.symlink(drive_outputs, f"{local_project}/outputs")

print("✅ Çalışma ortamı hazır!")

### 3. Önbellek Paketini Açma (`autsl_10_caches.zip`)
*Drive'daki önbellek paketini Colab'ın ultra hızlı yerel SSD'sine açar.*

In [ ]:
import glob, os

local_cache_root = "/content/BitirmeProjesi/cache"
os.makedirs(local_cache_root, exist_ok=True)

# Drive'daki autsl_10_caches.zip dosyasını otomatik bul
found_cache = glob.glob("/content/drive/MyDrive/**/autsl_10_caches.zip", recursive=True)

if found_cache:
    cache_zip = found_cache[0]
    print(f"📦 Önbellek paketi bulundu: {cache_zip}")
    # Zaten açılmış mı kontrol et
    strict_val_count = len(glob.glob("/content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict/val/*.npz"))
    if strict_val_count == 199:
        print(f"⚡ Önbellek zaten yerel SSD'de mevcut (199 val dosyası tam)! Tekrar açmaya gerek yok.")
    else:
        print("🚀 Önbellek yerel SSD'ye açılıyor (~10 saniye)...\n")
        !unzip -q -o "{cache_zip}" -d {local_cache_root}
        print("✅ Önbellek başarıyla açıldı!")
else:
    print("❌ HATA: autsl_10_caches.zip Drive'da bulunamadı! Lütfen Drive'a yüklediğinizden emin olun.")

### 4. 🚀 Pose-Only Modelini Eğitme (`autsl_10_poseonly_seed7`)
- Model: Yalnızca 225 Landmark Noktası (El CNN devre dışı)
- 20 Epoch, Seed 7, Cosine Annealing, AdamW
- T4 GPU ile yaklaşık **3-4 dakika** sürer.

In [ ]:
%cd /content/BitirmeProjesi
import os

# Metadata dizinini belirle
meta_dir = "/content/BitirmeProjesi/data/autsl"
if not os.path.exists(os.path.join(meta_dir, "val_labels.csv")):
    meta_dir = "/content/drive/MyDrive/BitirmeProjesi/outputs/colab_training_metadata"
os.environ["AUTSL_DATA_DIR"] = meta_dir

print("🚀 Pose-Only Model Eğitimi Başlatılıyor...\n")
!python scripts/train.py \
    --config configs/experiment/autsl_10_poseonly_seed7.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict

print("\n✅ Eğitim başarıyla tamamlandı!")

### 5. 📊 Resmi Val Seti Değerlendirmesi ve Video Tahminleri
*199 val videosu üzerinde değerlendirme yapar ve video bazlı tahminleri kaydeder (~20 saniye).* 

In [ ]:
%cd /content/BitirmeProjesi

!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_poseonly_seed7.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_poseonly_seed7/checkpoints/best_model.pt \
    --split val \
    --output outputs/poseonly_seed7_val.json \
    --predictions_output outputs/poseonly_seed7_predictions.json

print("\n✅ Değerlendirme ve tahmin kayıtları tamamlandı!")

### 6. 🔍 Teşhis Analizi: Hibrit vs Pose-Only (İşaretçi 11 Nerede?)

In [ ]:
%cd /content/BitirmeProjesi

# İkili eşleştirme analizi
!python scripts/analyze_val_pairs.py \
    --base outputs/centerrepair_seed7_predictions.json \
    --candidate outputs/poseonly_seed7_predictions.json \
    --official_labels "$AUTSL_DATA_DIR/val_labels.csv" \
    --output outputs/hybrid_vs_poseonly_seed7_pairs.json

import os, json
import pandas as pd

p_path = "outputs/poseonly_seed7_val.json"
c_path = "outputs/centerrepair_seed7_val.json"

if os.path.exists(p_path) and os.path.exists(c_path):
    p_res = json.load(open(p_path))
    c_res = json.load(open(c_path))
    
    df = pd.DataFrame([
        {"Model Mimarisi": "1. Hibrit Model (RGB El Kırpımı + İskelet)", "Val Doğruluk": f"%{c_res['accuracy']*100:.2f}", "Macro F1": f"{c_res['f1_macro']:.4f}", "Top-5": f"%{c_res.get('top5_accuracy', 0)*100:.2f}"},
        {"Model Mimarisi": "2. Pose-Only (Sadece İskelet Noktaları)", "Val Doğruluk": f"%{p_res['accuracy']*100:.2f}", "Macro F1": f"{p_res['f1_macro']:.4f}", "Top-5": f"%{p_res.get('top5_accuracy', 0)*100:.2f}"}
    ])
    display(df)
else:
    print("Sonuç dosyaları bekleniyor...")

### 7. 📦 Çıktıların Otomatik Paketlenmesi (ChatGPT İçin)
*ChatGPT'ye göndereceğiniz tüm dosyaları tek bir zip haline getirir.*

In [ ]:
import os, zipfile

zip_out = "/content/drive/MyDrive/BitirmeProjesi/chatgptye_gonderilecekler_poseonly.zip"

files_to_pack = {
    "poseonly_seed7_val.json": "/content/BitirmeProjesi/outputs/poseonly_seed7_val.json",
    "poseonly_seed7_predictions.json": "/content/BitirmeProjesi/outputs/poseonly_seed7_predictions.json",
    "hybrid_vs_poseonly_seed7_pairs.json": "/content/BitirmeProjesi/outputs/hybrid_vs_poseonly_seed7_pairs.json",
    "poseonly_seed7_history.csv": "/content/BitirmeProjesi/outputs/autsl_10_poseonly_seed7/checkpoints/history.csv",
    "poseonly_seed7_curves.png": "/content/BitirmeProjesi/outputs/autsl_10_poseonly_seed7/checkpoints/training_curves.png",
}

with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    for arcname, src_path in files_to_pack.items():
        if os.path.exists(src_path):
            zf.write(src_path, arcname=arcname)
            print(f"✅ Pakete eklendi: {arcname}")
        else:
            print(f"⚠️ Bulunamadı: {src_path}")

print(f"\n🎉 TEBRİKLER! Tek parça paket Drive'da hazır:\n{zip_out}")